This module codes the entire GPT architecture, including
* Layer Normalization
* Feedforward Network with GELU activation
* Shortcut connections
* Transformer module, including the Self Attention Mechanism
* Generating outputs from logits

# 0. Config for our model

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# 1. Creating a DummyGPT Model

We use placeholders for the transformer, layer normalization, feed-forward modules and so on, that we'll fill in during later sections

In [13]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Placeholder TransformerBlocks
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Placeholder LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, input):
        batch_size, seq_len = input.shape
        tok_embeds = self.tok_emb(input)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=input.device))
        x = tok_embeds + pos_embeds

        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Implementation later in the notebook

    def forward(self, x):
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Implementation later in the notebook

    def forward(self, x):
        return x

![Types of Attention](images/gpt-architecture-1.png)

In [11]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [14]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.9289,  0.2748, -0.7557,  ..., -1.6070,  0.2702, -0.5888],
         [-0.4476,  0.1726,  0.5354,  ..., -0.3932,  1.5285,  0.8557],
         [ 0.5680,  1.6053, -0.2155,  ...,  1.1624,  0.1380,  0.7425],
         [ 0.0447,  2.4787, -0.8843,  ...,  1.3219, -0.0864, -0.5856]],

        [[-1.5474, -0.0542, -1.0571,  ..., -1.8061, -0.4494, -0.6747],
         [-0.8422,  0.8243, -0.1098,  ..., -0.1434,  0.2079,  1.2046],
         [ 0.1355,  1.1858, -0.1453,  ...,  0.0869, -0.1590,  0.1552],
         [ 0.1666, -0.8138,  0.2307,  ...,  2.5035, -0.3055, -0.3083]]],
       grad_fn=<UnsafeViewBackward0>)


# 2. Layer Normalization

## Why is this needed?

While training our neural network, we can run into problems like exploding or vanishing gradients during the learning process.

Layer Normalization improves the (a) stability and (b) efficiency of our neural network training by adjusting the activations (outputs) of a given layer to have a mean of zero and a variance of 1.

## How do we do it?

We subtract the mean from each entry and divide each by the square root of variance.
Eventually, we want the vector corresponding to each input to have
* mean = 0
* variance = 1

## At what level are we normalizing?

We will be normalizing:
* each output independently
* along the last dimension of each output row, which will generally be the embedding dimension (`emb_dim`).

Let's say the hidden state dimension is [batch_size, seq_len, emb_dim], such as [2, 4, 768]. We normalize each of the 4 tokens (rows) independently across the 768 features. Hence, for each of the two batches:
* row1 => mean = 0, variance = 1
* row2 => mean = 0, variance = 1
* row3 => mean = 0, variance = 1
* row4 => mean = 0, variance = 1

## Layer norm vs Batch norm?

Batch normalziation normalizes across the batches (that is, on the batch dimension), whereas LN normalizes across the feature dimensions.

Since LN normalizes each output/input independent of the back size, it offers more flexibility and stability, specially during distributed training or when deploying models in environments where resources are constrained.

## Where is Layer Normalization applied?

* Before and after the MHA module in the transformer module
* Before the final output layer

## Pre vs Post Layer Normalization?

TODO

## Can you show me a diagram?

![](images/gpt-architecture-2.png)

## Understanding Layer Normalization

In [43]:
# Let's build a simple NN with 5 inputs and 6 outputs to understand LN

torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5)

# simple NN with 5 inputs, 6 outputs and ReLU activation
dummy_NN_layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())

input = dummy_NN_layer(batch_example)
print(output)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [44]:
# Computing the mean and variance of the above input

def calc_mean_and_var(x):
    # dim = -1 asks for computing mean along the column dimension, hence,
    # we obtain one mean per row
    
    # keepdim=True is to preserve the dimensionality of the input tensor,
    # even though our operation reduces the tensor
    # along the dimension specified by dim
    # for example, our output would've been a 2-D vector [0.1324, 0.2170], instead
    # of 2 x 1-dimensional matrix [[0.1324], [0.2170]]
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True)
    
    print("Mean: \n", mean)
    print("Variance: \n", var)

calc_mean_and_var(output)

Mean: 
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [45]:
# applying Layer Norm to the outputs we received earlier from our dummy NN

output_normalized = (output - mean) / torch.sqrt(var)
print("Normalized layer outputs: \n", output_normalized)

calc_mean_and_var(output_normalized)

Normalized layer outputs: 
 tensor([[2.2601e-01, 3.4704e-01, 0.0000e+00, 2.2166e-01, 0.0000e+00, 0.0000e+00],
        [2.1332e-01, 2.3946e-01, 1.9871e-08, 5.1991e-01, 3.2980e-01, 1.9871e-08]],
       grad_fn=<DivBackward0>)
Mean: 
 tensor([[0.1325],
        [0.2171]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


## Layer Normalization Class

In [49]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        # there is some deets around why unbiased = False, but we 
        # can ignore it for now. Just know that GPT-2 was also
        # trained with a biased variance in normalization layers
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_x = (x - mean) / torch.sqrt(var + self.eps)

        output = self.scale * normalized_x + self.shift

        return output

## What does this class normalize?

The class is designed to operate on the last dimension of our input tensor x, which represents the embedding dimension (`emb_dim`).

## What are scale and shift?

The initial scale (multiplying by 1) and shift (adding 0) values don't have any effect; however, scale and shift are trainable parameters that the LLM automatically adjusts during training if it is determined that doing so would improve the model's performance on its training task
This allows the model to learn appropriate scaling and shifting that best suit the data it is processing.

In [46]:
# Now try the class on the sample data we had constructed before

ln = LayerNorm(emb_dim=6)
normalized_output = ln(output)

In [48]:
mean = normalized_output.mean(dim=-1, keepdim=True)
var = normalized_output.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

# Variance is not exactly 1 because we use eps

Mean:
 tensor([[ 0.0000e+00],
        [-1.9868e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.9995],
        [0.9997]], grad_fn=<VarBackward0>)
